In [0]:
# To create Text widget to get source file path
# dbutils.widgets.text("Source Folder Path", "")

In [0]:
%pip install databricks-feature-engineering
%pip install xgboost

In [0]:
dbutils.library.restartPython()

In [0]:
source_folder_path = dbutils.widgets.get("Source Folder Path")
if not source_folder_path:
    dbutils.notebook.exit("source_folder_path is not provided.")

In [0]:
from databricks.feature_store import FeatureStoreClient, FeatureLookup
from sklearn.model_selection import train_test_split

In [0]:
# Create feature Lookups

feature_lookups = [
    FeatureLookup(
        table_name='workspace.sales.customer_features',
        feature_names=['total_purchase_7d', 'total_purchase_30d'],
        lookup_key='customer_id'
    ),
    FeatureLookup(
        table_name='workspace.sales.product_features',
        feature_names=['category'],
        lookup_key='product_id'
    )
]

In [0]:
label_file = f"{source_folder_path}/training_labels.csv"
label_df = spark.read.load(label_file, format="csv",sep=",",inferSchema="true",header="true" )

display(label_df)

In [0]:
fs = FeatureStoreClient()
def load_data(label_df, feature_lookups):
 
    # fs.create_training_set looks up features in model_feature_lookups that match the primary key from inference_data_df
    training_set = fs.create_training_set(label_df, feature_lookups, label="purchased", exclude_columns=[ 'customer_id' , 'product_id'])

    training_pd = training_set.load_df().toPandas()
 
    # Create train and test datasets
    X = training_pd.drop("purchased", axis=1)
    y = training_pd["purchased"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    return X_train, X_test, y_train, y_test, training_set

In [0]:
# Create the train and test datasets
X_train, X_test, y_train, y_test, training_set = load_data(label_df, feature_lookups)
X_train.head()

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.tracking.client import MlflowClient
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [0]:
DB = "sales"

In [0]:

 
client = MlflowClient()
 
try:
    client.delete_registered_model(f"{DB}.purchase_model") # Delete the model if already created
except:
    None

In [0]:
X_train['category'] = X_train['category'].astype('category')
X_test['category'] = X_test['category'].astype('category')

In [0]:
mlflow.sklearn.autolog(log_models=False)
 
def train_model(X_train, X_test, y_train, y_test, training_set, fs):
    ## fit and log model
    with mlflow.start_run() as run:
        model = XGBClassifier(
            max_depth=3,
            n_estimators=20,
            random_state=42,
            enable_categorical=True,
            tree_method="hist"
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log standard classification metrics
        mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_pred))
        mlflow.log_metric("test_precision", precision_score(y_test, y_pred))
        mlflow.log_metric("test_recall", recall_score(y_test, y_pred))
        mlflow.log_metric("test_f1_score", f1_score(y_test, y_pred))



        # Log model with Feature Store
        fs.log_model(
            model=model,
            artifact_path="purchase_prediction_model",
            flavor=mlflow.sklearn,
            training_set=training_set,
            registered_model_name=f"{DB}.purchase_model"
        )
 
train_model(X_train, X_test, y_train, y_test, training_set, fs)